# M6 · Lab 3 — Great Expectations Data Validation
### Module 6 — Monitoring, Testing & Drift Detection | Spine Project: Truck Delay Classification

| | |
|---|---|
| **Duration** | 60 min |
| **Difficulty** | Intermediate |
| **Tools** | Python 3.12.10, `great-expectations>=0.18,<1.0`, pandas |
| **AWS** | None (local notebook — Lab 4 wires GE's pass/fail into SNS) |
| **Prerequisite** | Lab 2 (same `data/` folder with the real M3 reference frame) |
| **Builds toward** | Lab 4 + Lab 5 (combined pipeline runs GE → Evidently → SNS) |
| **Cost** | ₹0 — all local |

---
Lab 2 caught **statistical** drift. This lab catches **schema** breaks — corrupted values, NULL spikes, type changes,
out-of-range numbers. Drift is gradual and probabilistic; schema breaks are sudden and binary. Production needs **both**.

## 💻 Where to run this — SageMaker (recommended in class), Colab, or local

**In class: the SageMaker Notebook Instance your instructor provisioned** (`m6-truck-delay-monitoring`). Everything is
pre-installed (`evidently`, `great-expectations`, `xgboost`) and the real `data/` folder is already there — just open this
notebook and run.
- **Kernel:** `conda_python3`
- **Data:** `data/reference/final_features.csv` + `data/artifacts/` ship with the labs on the instance (the `M6_DATA_DIR` default is `data`).
- **Auto-stop:** the instance stops itself after ~45 min idle to save cost — your work, installed packages, and saved
  artifacts persist on the EBS volume. Just hit **Start** the next day; **M7 and M8 reuse this same instance.**
- **AWS access (Labs 4/5):** the instance's IAM role already grants `sns:Publish`, so there are **no access keys to configure.**

**Portable fallback (self-paced / post-course):** this notebook also runs on **Google Colab** or **local Jupyter** — the
setup cell below detects the environment and pip-installs what's missing. On Colab, upload the `data/` folder (or set
`M6_DATA_DIR`). Locally, use **Python 3.12.10**. The data + model are the *real* M3 artifacts either way — nothing synthetic.

## Learning Objectives
1. Explain what **Great Expectations (GE)** solves and where the GE/Evidently line sits.
2. Auto-profile the **real** reference frame into a starting **expectation suite**.
3. Hand-edit the suite to add domain rules the profiler can't infer.
4. Validate a **clean** batch (100% pass) and a **corrupted** batch (specific row-level failures).
5. Emit a structured failure payload + generate **Data Docs** (the team's "what must the data look like?" contract).

## Business Context
A month into production, the upstream scheduling service started emitting `route_id` as the string `"R-04522"` instead
of the integer `4522`. The Truck Delay app didn't crash — it coerced the bad value to NaN, produced a low-confidence
prediction, and operations treated it as "on time". **Three days of misrouted dispatches** before anyone joined the dots.

That's **not drift** — a column of NaNs has no distribution to compare. It's a **schema break**, and it needs *declarative
validation*: you write down "`truck_age` is an integer in [1, 30]" and every incoming batch is checked against the
contract. One violating value fails the batch — loudly, with a row-level message — before the bad data reaches the model.

## Prerequisites — the **real** M3 reference frame (no synthetic data)
Same `data/` folder as Lab 2. We profile the genuine training distribution — **`data/reference/final_features.csv`**
(12,308 × 37, regenerated from M3 Lab B and round-trip-validated against the model). The *only* synthetic thing here is
the deliberately **corrupted batch** in Step 7 — you can't demonstrate catching corruption without some.

> **Why pin GE `<1.0`?** GE v1.0 (Sep 2024) is a breaking rewrite. The course — and the Lab 5 production pipeline that
> consumes this suite — uses the stable **0.18** line. Once you know 0.18, migrating to v1 is a half-day.

In [1]:
# ── Environment setup (Colab or local) ────────────────────────────────────────
import sys, subprocess, os
def _pip(*p): subprocess.run([sys.executable, "-m", "pip", "install", "-q", *p], check=True)
try:
    import great_expectations as gx
    print("great_expectations:", gx.__version__)
except ImportError:
    print("Installing great-expectations 0.18 …")
    _pip("great-expectations>=0.18,<1.0", "pandas==2.2.0", "numpy<2.0")
    import great_expectations as gx
    print("great_expectations:", gx.__version__)

great_expectations: 0.18.22


In [2]:
# ── Imports + load the real reference frame ───────────────────────────────────
import json
import numpy as np
import pandas as pd

DATA_DIR = os.environ.get("M6_DATA_DIR", "data")
REF_CSV  = os.path.join(DATA_DIR, "reference", "final_features.csv")
assert os.path.exists(REF_CSV), f"Real reference frame missing at {REF_CSV} (copy Module 6/labs/data/ next to this notebook)."

ref = pd.read_csv(REF_CSV)
print(f"Reference frame: {ref.shape}  (expected 12308 x 37)")
ref.head(3)

Reference frame: (12308, 37)  (expected 12308 x 37)


,route_avg_temp,route_avg_wind_speed,route_avg_precip,route_avg_humidity,route_avg_visibility,route_avg_pressure,origin_avg_temp,origin_avg_wind_speed,origin_avg_precip,origin_avg_humidity,...,route_description,origin_description,dest_description,accident,fuel_type,gender,driving_style,ratings,is_midnight,delay
0,31.00,4.25,0.0,86.00,6.0,1009.50,26.041667,10.5,0.0,80.625,...,Clear,Light snow,Overcast,0,diesel,male,proactive,7,0,0
1,43.25,9.25,0.0,78.50,6.0,1021.25,26.041667,10.5,0.0,80.625,...,Partly cloudy,Light snow,Sunny,1,diesel,male,proactive,8,1,0
2,24.00,7.00,0.0,63.75,5.5,1019.75,26.041667,10.5,0.0,80.625,...,Clear,Light snow,Moderate snow,0,diesel,male,proactive,8,0,0


## Step 1 · Concepts — what's an "Expectation"?
A single **expectation** is one assertion about a column:
```python
expect_column_values_to_be_between("truck_age", min_value=1, max_value=30)
```
There are ~50 built-ins:

| Category | Example | Checks |
|---|---|---|
| Schema | `expect_column_to_exist` | the column is present |
| Type | `expect_column_values_to_be_of_type` | int / float / str |
| Range | `expect_column_values_to_be_between` | min/max bounds |
| Set | `expect_column_values_to_be_in_set` | value ∈ enum |
| Null | `expect_column_values_to_not_be_null` | NULL rate (`mostly=0.99`) |
| Cardinality | `expect_column_distinct_values_to_be_in_set` | exact allowed set |

An **expectation suite** is a JSON file of dozens of these. A **checkpoint** runs a suite against new data.
Flow: **profile → edit → validate → publish**.

## Step 2 · Create a file-backed GE project + register the reference as a runtime batch

In [3]:
import os
from great_expectations.data_context import FileDataContext
from great_expectations.core.batch import RuntimeBatchRequest

# Always create/load a FILE-backed context that persists to ./great_expectations/.
# The old try/except could silently fall back to an in-memory (Ephemeral) context,
# which makes every save_expectation_suite() succeed but write nothing to disk --
# that's why Lab 4 then failed with "expectation_suite truck_delay_features not found".
if os.path.isdir("great_expectations"):
    context = gx.get_context(context_root_dir="./great_expectations")   # load existing project
else:
    context = FileDataContext.create(project_root_dir=".")              # create it the first time

print("context type :", type(context).__name__)        # MUST print FileDataContext (not Ephemeral...)
print("GE project root:", context.root_directory)
assert type(context).__name__ == "FileDataContext", (
    "Got an in-memory context -- suites will NOT persist. Ensure this folder is writable "
    "and re-run. (Expected FileDataContext.)"
)

# A runtime datasource lets us validate in-memory pandas DataFrames (the production pattern).
if "truck_delay_runtime" not in [ds["name"] for ds in context.list_datasources()]:
    context.add_datasource(
        name="truck_delay_runtime",
        class_name="Datasource",
        execution_engine={"class_name": "PandasExecutionEngine"},
        data_connectors={
            "default_runtime_data_connector_name": {
                "class_name": "RuntimeDataConnector",
                "batch_identifiers": ["default_identifier_name"],
            }
        },
    )
print("Datasources:", [ds["name"] for ds in context.list_datasources()])


context type : FileDataContext
GE project root: /home/sagemaker-user/MLOpsself/Module 6 complete/labs/gx
Datasources: ['truck_delay_runtime']


## Step 3 · Build the SCHEMA CONTRACT (structure, not distribution)

We deliberately do **not** auto-profile distributional bounds (`expect_column_mean/min/max_to_be_between`,
value-sets on continuous columns). Those are fitted to the reference, so a legitimate distribution shift — monsoon
precip/humidity, fleet aging — would trip **GE** as a "schema break" *before* the Evidently drift detector ever runs
(that's exactly what stopped Lab 5's `--simulate` at the GE stage). A schema contract asserts **structure**: the columns
are present, there are no NULL spikes, the stable categoricals stay in their allowed set, the target is binary, and
`truck_age` is in its operating range. Distribution is Evidently's job (Lab 2), not GE's.

In [4]:
SUITE_NAME = "truck_delay_features"
context.add_or_update_expectation_suite(expectation_suite_name=SUITE_NAME)   # reset to empty, then add

def batch_request_for(df, label):
    '''Wrap an in-memory DataFrame as a GE RuntimeBatchRequest.'''
    return RuntimeBatchRequest(
        datasource_name="truck_delay_runtime",
        data_connector_name="default_runtime_data_connector_name",
        data_asset_name="truck_delay",
        runtime_parameters={"batch_data": df},
        batch_identifiers={"default_identifier_name": label},
    )

v = context.get_validator(
    batch_request=batch_request_for(ref, "schema_build"),
    expectation_suite_name=SUITE_NAME,
)

# 1) All expected columns are present (the column contract)
v.expect_table_columns_to_match_set(column_set=list(ref.columns), exact_match=False)

# 2) No NULL spikes — assert not-null on every column that is fully populated in the reference.
#    (This is what catches the corrupted batch's null `ratings` in Step 7.)
for col in ref.columns:
    if ref[col].notna().all():
        v.expect_column_values_to_not_be_null(col)

# 3) Stable categoricals must stay within the training set.
#    We EXCLUDE the weather *_description columns on purpose: new rain categories in monsoon are
#    DRIFT (Evidently's job), not a schema break — constraining them here would make GE pre-empt drift.
STABLE_CATEGORICALS = ["fuel_type", "gender", "driving_style", "is_midnight", "accident"]
for col in STABLE_CATEGORICALS:
    if col in ref.columns:
        v.expect_column_values_to_be_in_set(col, value_set=sorted(ref[col].dropna().unique().tolist()))

v.save_expectation_suite(discard_failed_expectations=False)
n_struct = len(context.get_expectation_suite(SUITE_NAME).expectations)
print(f"Schema contract built: {n_struct} structural expectations across {ref.shape[1]} columns")


Calculating Metrics:   0%|          | 0/2 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/6 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/6 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/6 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/6 [00:00<?, ?it/s]

...

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Schema contract built: 43 structural expectations across 37 columns


## Step 4 · Hand-edit — add the domain rules the profiler can't infer
The profiler writes *empirical* bounds (e.g. `truck_age ∈ [1, 23]` because that's what it saw). We widen them to the
*business* range so next year's 24-year-old truck doesn't trip a false alarm — and we add cross-column + enum rules.

In [5]:
v = context.get_validator(
    batch_request=batch_request_for(ref, "domain_rules"),
    expectation_suite_name=SUITE_NAME,
)

# Business RANGE for truck_age: widen to the operating range [1, 30] (NOT the empirical [1, 23] the
# data happens to show) so next year\'s older trucks don\'t false-alarm — while still catching impossible
# values like the -3 injected in Step 6.
v.expect_column_values_to_be_between("truck_age", min_value=1, max_value=30, mostly=1.0)

# Target must be binary (catches the delay=2 corruption in Step 6)
v.expect_column_values_to_be_in_set("delay", value_set=[0, 1])
v.expect_column_distinct_values_to_be_in_set("delay", value_set=[0, 1])

# NOTE: we do NOT add `age >= experience` here. The real M3 frame contains a few negative
# `experience` values (a known upstream quirk), so that cross-column rule would fail the CLEAN
# reference — a schema contract must pass healthy data. Catching bad experience belongs in M3 cleaning.

v.save_expectation_suite(discard_failed_expectations=False)
suite = context.get_expectation_suite(SUITE_NAME)
print(f"Suite now has {len(suite.expectations)} expectations (schema contract + business range/target rules)")


Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/4 [00:00<?, ?it/s]

Suite now has 46 expectations (schema contract + business range/target rules)


## Step 5 · Validate a CLEAN batch — the reference itself should pass ~100%

In [6]:
res_clean = context.get_validator(
    batch_request=batch_request_for(ref, "clean_validation"),
    expectation_suite_name=SUITE_NAME,
).validate()

print(f"Success:      {res_clean['success']}")
print(f"Expectations: {res_clean['statistics']['successful_expectations']}"
      f" / {res_clean['statistics']['evaluated_expectations']}")
print(f"Success %:    {res_clean['statistics']['success_percent']:.1f}")
# If <100%, a hand-edit over-tightened — loosen the offending range and re-run Step 4.

Calculating Metrics:   0%|          | 0/136 [00:00<?, ?it/s]

Success:      True
Expectations: 46 / 46
Success %:    100.0


## Step 6 · Build a deliberately CORRUPTED batch (the only synthetic part)

In [7]:
def make_corrupt_batch(reference: pd.DataFrame, n: int = 500, seed: int = 7) -> pd.DataFrame:
    '''Sample real rows and inject four classes of corruption GE should catch.'''
    b = reference.sample(n=n, random_state=seed).reset_index(drop=True).copy()
    b.loc[b.index[:25], "truck_age"] = -3                         # 1) out-of-range (negative)
    null_idx = b.sample(frac=0.05, random_state=1).index
    b.loc[null_idx, "ratings"] = np.nan                           # 2) NULL spike
    b.loc[b.index[300:310], "fuel_type"] = "hydrogen"             # 3) unknown category
    b.loc[b.index[400:402], "delay"] = 2                          # 4) target out of {0,1}
    return b

corrupt = make_corrupt_batch(ref)
print(f"Corrupted batch: {corrupt.shape}  (negative truck_age, null ratings, 'hydrogen' fuel, delay=2)")

Corrupted batch: (500, 37)  (negative truck_age, null ratings, 'hydrogen' fuel, delay=2)


## Step 7 · Validate the corrupted batch — GE should fail with specific row-level errors

In [8]:
res_bad = context.get_validator(
    batch_request=batch_request_for(corrupt, "corrupted_validation"),
    expectation_suite_name=SUITE_NAME,
).validate()

print(f"Success: {res_bad['success']}")
print(f"Failed:  {res_bad['statistics']['unsuccessful_expectations']}"
      f" / {res_bad['statistics']['evaluated_expectations']}\n")
print("Failures:")
for r in res_bad["results"]:
    if not r["success"]:
        e = r["expectation_config"]
        col = e["kwargs"].get("column", e["kwargs"].get("column_A", "<table>"))
        unexpected = r["result"].get("unexpected_count", "?")
        sample = r["result"].get("partial_unexpected_list", [])[:3]
        print(f"  X {e['expectation_type']:42s} col={col:14s} unexpected={unexpected}  sample={sample}")

Calculating Metrics:   0%|          | 0/136 [00:00<?, ?it/s]

Success: False
Failed:  5 / 46

Failures:
  X expect_column_values_to_be_between         col=truck_age      unexpected=25  sample=[-3, -3, -3]
  X expect_column_values_to_be_in_set          col=fuel_type      unexpected=10  sample=['hydrogen', 'hydrogen', 'hydrogen']
  X expect_column_values_to_not_be_null        col=ratings        unexpected=25  sample=[nan, nan, nan]
  X expect_column_values_to_be_in_set          col=delay          unexpected=2  sample=[2, 2]
  X expect_column_distinct_values_to_be_in_set col=delay          unexpected=?  sample=[]


## Step 8 · The structured failure payload (what Lab 4 / Lab 5 publish to SNS)

In [9]:
def ge_failure_payload(results) -> dict:
    '''Reduce a GE validation result to the compact dict the alerting pipeline sends.'''
    return {
        "success": bool(results["success"]),
        "evaluated": results["statistics"]["evaluated_expectations"],
        "passed": results["statistics"]["successful_expectations"],
        "failed": results["statistics"]["unsuccessful_expectations"],
        "success_percent": round(results["statistics"]["success_percent"], 1),
        "failures": [
            {
                "expectation": r["expectation_config"]["expectation_type"],
                "column": r["expectation_config"]["kwargs"].get("column"),
                "unexpected_count": r["result"].get("unexpected_count", 0),
                "sample": r["result"].get("partial_unexpected_list", [])[:3],
            }
            for r in results["results"] if not r["success"]
        ],
    }

import pprint; pprint.pp(ge_failure_payload(res_bad))

{'success': False,
 'evaluated': 46,
 'passed': 41,
 'failed': 5,
 'success_percent': 89.1,
 'failures': [{'expectation': 'expect_column_values_to_be_between',
               'column': 'truck_age',
               'unexpected_count': 25,
               'sample': [-3, -3, -3]},
              {'expectation': 'expect_column_values_to_be_in_set',
               'column': 'fuel_type',
               'unexpected_count': 10,
               'sample': ['hydrogen', 'hydrogen', 'hydrogen']},
              {'expectation': 'expect_column_values_to_not_be_null',
               'column': 'ratings',
               'unexpected_count': 25,
               'sample': [nan, nan, nan]},
              {'expectation': 'expect_column_values_to_be_in_set',
               'column': 'delay',
               'unexpected_count': 2,
               'sample': [2, 2]},
              {'expectation': 'expect_column_distinct_values_to_be_in_set',
               'column': 'delay',
               'unexpected_count': 0,
       

## Step 9 · Generate Data Docs — the team's HTML schema contract

In [10]:
context.build_data_docs()
docs = context.get_docs_sites_urls()
print("Data Docs built:")
for d in docs:
    print("  ", d["site_url"])
# Open that file:// URL in a browser. Expectations tab = the contract; Validation Results tab = run history.

Data Docs built:
   file:///home/sagemaker-user/MLOpsself/Module 6 complete/labs/gx/uncommitted/data_docs/local_site/index.html


## Step 10 · Artifact chain — what downstream labs consume
The committed output is **`great_expectations/expectations/truck_delay_features.json`** plus the project around it.
**Lab 4** (combined exploration) and **Lab 5** (production script) load this exact suite by name and run it against each
incoming batch *before* the Evidently drift check (GE is cheap → fail fast on schema).

In [12]:
suite_path = os.path.join(context.root_directory, "expectations", f"{SUITE_NAME}.json")
print("Suite persisted at:", suite_path, "->", os.path.exists(suite_path))
print("GE project root (pass THIS to Lab 4 / Lab 5):", context.root_directory)

Suite persisted at: /home/sagemaker-user/MLOpsself/Module 6 complete/labs/gx/expectations/truck_delay_features.json -> True
GE project root (pass THIS to Lab 4 / Lab 5): /home/sagemaker-user/MLOpsself/Module 6 complete/labs/gx


## GE vs Evidently — where to draw the line
| | **Great Expectations** (this lab) | **Evidently** (Lab 2) |
|---|---|---|
| Question | "does this batch satisfy the schema contract?" | "is the distribution shifting over time?" |
| Fires when | a *single* row violates a rule | statistical distance crosses a threshold |
| Best for | schema breaks, type changes, NULL spikes, out-of-range | slow drifts, monsoon precip shift, fleet aging |
| Misses | subtle statistical shifts | an all-NULL column (degenerate distribution) |
| Cost | cheap (iterate rules) | ~1 s (Wasserstein on 30 cols × 1k rows) |

**Rule of thumb:** if it would *break your code* → GE catches it. If it would *degrade your accuracy* → Evidently catches it.

## Verification Checklist
- [ ] `great_expectations/` project created; `expectations/truck_delay_features.json` exists
- [ ] Clean validation (reference) reports ~100% pass
- [ ] Corrupted validation fails, naming `truck_age`, `ratings`, `fuel_type`, `delay`
- [ ] `ge_failure_payload(...)` returns the compact failure dict
- [ ] Data Docs HTML opens and shows the suite + both validation runs
- [ ] You can state the GE-vs-Evidently difference in two sentences

## What's next — Lab 4
You now have two independent monitors: an Evidently drift detector (Lab 2) and a GE validator (Lab 3). **Lab 4** fuses
them into one flow — GE first (fail fast), then Evidently, then publish a structured alert to the **SNS topic from Lab 1**
— and returns an exit code so it composes with cron / Airflow / EventBridge. **Lab 5** ships that flow as a production `.py`.

## Troubleshooting
| Symptom | Fix |
|---|---|
| `ImportError: UserConfigurableProfiler` | GE v1 installed — pin `great-expectations>=0.18,<1.0` |
| `len(suite.expectations) == 0` | the validator wasn't fed a real batch — confirm `runtime_parameters={"batch_data": ref}` |
| "no datasource named ..." on validate | re-run Step 2 (the runtime datasource must be registered in this kernel) |
| clean validation < 100% | a hand-edit over-tightened — widen the offending range in Step 4 |
| `FileDataContext.create` errors "already exists" | fine — the `except` branch loads the existing project |
| Data Docs "no validations to render" | run a validation (Step 5) before `build_data_docs()` |